# 03. Adherence Analysis and Data Quality Checks

This notebook focuses on operational quality checks, adherence metrics, and the future feature engineering handoff to Member 3.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

csv_path = repo_root / "ml" / "data" / "raw" / "sample_synthetic_medication_data.csv"
df = pd.read_csv(csv_path, comment="#")
print(df.head())

In [ ]:
# Data quality check functions

def is_valid_time(value):
    if pd.isna(value):
        return True
    try:
        if isinstance(value, str):
            datetime.strptime(value, "%H:%M:%S")
        return True
    except ValueError:
        try:
            datetime.strptime(str(value), "%H:%M")
            return True
        except ValueError:
            return False


def check_quality(df):
    issues = []

    # missing patient IDs
    missing_patient = df[df["patient_id"].isna() | (df["patient_id"].astype(str).str.strip() == "")]
    if not missing_patient.empty:
        issues.append(f"Missing patient ID rows: {len(missing_patient)}")

    # missing medicine IDs
    missing_medicine = df[df["medicine_id"].isna() | (df["medicine_id"].astype(str).str.strip() == "")]
    if not missing_medicine.empty:
        issues.append(f"Missing medicine ID rows: {len(missing_medicine)}")

    # invalid dose status
    valid_statuses = {"TAKEN", "PENDING", "MISSED"}
    invalid_status = df[~df["dose_status"].isin(valid_statuses)]
    if not invalid_status.empty:
        issues.append(f"Invalid dose status rows: {len(invalid_status)}")

    # invalid time formats
    invalid_times = []
    for col in ["scheduled_time", "reminder_time", "acknowledged_time"]:
        if col in df.columns:
            bad = df[df[col].notna() & ~df[col].map(is_valid_time)]
            if not bad.empty:
                invalid_times.append(f"{col}: {len(bad)}")
    if invalid_times:
        issues.extend(invalid_times)

    # negative delay values
    negative_delays = df[pd.to_numeric(df["delay_minutes"], errors="coerce") < 0]
    if not negative_delays.empty:
        issues.append(f"Negative delay rows: {len(negative_delays)}")

    # duplicate dose events
    duplicate_rows = df.duplicated(subset=["patient_id", "medicine_id", "scheduled_date", "scheduled_time"]).sum()
    if duplicate_rows > 0:
        issues.append(f"Duplicate dose events: {duplicate_rows}")

    # impossible timestamps
    if {"scheduled_date", "scheduled_time"}.issubset(df.columns):
        try:
            parsed = pd.to_datetime(df["scheduled_date"].astype(str) + " " + df["scheduled_time"].astype(str), errors="coerce")
            impossible = df[parsed.isna()]
            if not impossible.empty:
                issues.append(f"Impossible timestamps: {len(impossible)}")
        except Exception:
            issues.append("Timestamp parsing raised an exception.")

    # inconsistent schedule/actual times
    if {"scheduled_time", "acknowledged_time"}.issubset(df.columns):
        actual_numeric = pd.to_datetime(df["acknowledged_time"], format="%H:%M:%S", errors="coerce")
        schedule_numeric = pd.to_datetime(df["scheduled_time"], format="%H:%M:%S", errors="coerce")
        if not actual_numeric.isna().all():
            inconsistent = df[(actual_numeric.notna()) & (schedule_numeric.notna()) & (actual_numeric < schedule_numeric)]
            if not inconsistent.empty:
                issues.append(f"Inconsistent schedule/actual times: {len(inconsistent)}")

    return issues

quality_issues = check_quality(df)
print("Quality issues:")
if quality_issues:
    for issue in quality_issues:
        print(f"- {issue}")
else:
    print("No critical quality issues detected in this synthetic dataset.")

In [ ]:
# Adherence metrics

total_scheduled = len(df)
total_taken = (df["dose_status"] == "TAKEN").sum()
total_missed = (df["dose_status"] == "MISSED").sum()
total_pending = (df["dose_status"] == "PENDING").sum()
adherence_pct = (total_taken / total_scheduled * 100) if total_scheduled else 0.0

print({
    "total_scheduled_doses": total_scheduled,
    "total_taken_doses": total_taken,
    "total_missed_doses": total_missed,
    "total_pending_doses": total_pending,
    "overall_adherence_percentage": round(adherence_pct, 2),
})

## ML feature candidates and leakage risks

Potential future features include:

- `delay_minutes`
- `previous_missed_doses`
- `previous_taken_doses`
- `adherence_percentage`
- `time_period`
- `day_of_week`
- `frequency`
- `dose_quantity`
- `sensor_event`

Leakage risk: do not use `acknowledged_time` to predict if the same dose was taken when the value is only known after the event. Similar target leakage should be avoided for all outcome-derived features.

## Handoff to Member 3

This data-analysis work provides the foundation for the future ML team:

```text
Member 4
Data collection schema
        ↓
Data cleaning
        ↓
EDA
        ↓
Feature recommendations
        ↓
Member 3
ML model development
```

The next stage for Member 3 is:

```text
Feature Engineering
        ↓
Train/Test Split
        ↓
Baseline ML Model
        ↓
Model Evaluation
```